In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#load data usign google file upload
from google.colab import files
uploaded = files.upload()

Saving RAW_recipes.csv to RAW_recipes.csv


In [ ]:
import pandas as pd
data= pd.read_csv('RAW_recipes.csv')

In [ ]:
data.head()

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8


In [ ]:
import pandas as pd
import ast
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense


In [ ]:

# Drop rows with missing essential fields
df_clean = data.dropna(subset=["ingredients", "steps", "tags"])
df_clean = df_clean[df_clean["ingredients"].str.len() > 2]
df_clean = df_clean[df_clean["steps"].str.len() > 2]
df_clean = df_clean[df_clean["tags"].str.len() > 2]

# Convert stringified lists to actual Python lists
df_clean["ingredients"] = df_clean["ingredients"].apply(ast.literal_eval)
df_clean["steps"] = df_clean["steps"].apply(ast.literal_eval)
df_clean["tags"] = df_clean["tags"].apply(ast.literal_eval)

# Filter out empty lists
df_clean = df_clean[df_clean["ingredients"].apply(len) > 0]
df_clean = df_clean[df_clean["steps"].apply(len) > 0]
df_clean = df_clean[df_clean["tags"].apply(len) > 0]

# Convert to lowercase text strings
df_clean["ingredients_text"] = df_clean["ingredients"].apply(
    lambda x: ' '.join(i.lower() for i in x)
)
df_clean["tags_text"] = df_clean["tags"].apply(
    lambda x: ' '.join(t.lower().replace("-", " ") for t in x)
)
df_clean["steps_text"] = df_clean["steps"].apply(
    lambda x: ' '.join(s.lower() for s in x)
)
df_clean["steps_text"] = df_clean["steps_text"].apply(lambda x: "<start> " + x + " <end>")

# Final input string: ingredients + tags
df_clean["input_text"] = df_clean["ingredients_text"] + " " + df_clean["tags_text"]

# Select only the needed columns
final_df = df_clean[["input_text", "steps_text"]].reset_index(drop=True)

# Save to file (optional)
final_df.to_csv("preprocessed_recipes.csv", index=False)

print(final_df.head())


                                          input_text  \
0  winter squash mexican seasoning mixed spice ho...   
1  prepared pizza crust sausage patty eggs milk s...   
2  ground beef yellow onions diced tomatoes tomat...   
3  spreadable cheese with garlic and herbs new po...   
4  tomato juice apple cider vinegar sugar salt pe...   

                                          steps_text  
0  make a choice and proceed with recipe dependin...  
1  preheat oven to 425 degrees f press dough into...  
2  brown ground beef in large pot add chopped oni...  
3  place potatoes in a large pot of lightly salte...  
4  mix all ingredients& boil for 2 1 / 2 hours , ...  


Metric -BLEU

BPE Tokenization for better performance

In [1]:
!pip install sentencepiece

In [20]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("preprocessed_recipes.csv")
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

# Then use train_df for BPE training
with open("bpe_corpus.txt", "w", encoding="utf-8") as f:
    for text in train_df["input_text"]:
        f.write(text.lower() + "\n")
    for text in train_df["steps_text"]:
        f.write(text.lower() + "\n")


In [21]:
#Train BPE tokenizer using SentencePiece
import sentencepiece as spm

spm.SentencePieceTrainer.Train(
    input='bpe_corpus.txt',
    model_prefix='bpe_tokenizer',
    vocab_size=8000,
    model_type='bpe',  # options: 'bpe', 'unigram', 'char', 'word'
    pad_id=0,
    bos_id=1,
    eos_id=2,
    unk_id=3
)

print("✅ BPE tokenizer trained: bpe_tokenizer.model and bpe_tokenizer.vocab generated")


✅ BPE tokenizer trained: bpe_tokenizer.model and bpe_tokenizer.vocab generated


In [22]:
# ✅ STEP 4: Encode and Decode Examples Using the Trained Model
import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.load("bpe_tokenizer.model")

# Test encoding
text = "add garlic and onion to the pan"
encoded = sp.encode(text, out_type=int)
decoded = sp.decode(encoded)

print("Encoded:", encoded)
print("Decoded:", decoded)

Encoded: [85, 217, 27, 169, 26, 29, 207]
Decoded: add garlic and onion to the pan


In [23]:
# ✅ STEP 5: Encode full dataset (to use in training later)
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN_INPUT = 132
MAX_LEN_TARGET = 300

encoded_inputs = [sp.encode(t, out_type=int) for t in df["input_text"]]
encoded_targets = [sp.encode("<start> " + t + " <end>", out_type=int) for t in df["steps_text"]]

input_padded = pad_sequences(encoded_inputs, maxlen=MAX_LEN_INPUT, padding='post')
target_padded = pad_sequences(encoded_targets, maxlen=MAX_LEN_TARGET, padding='post')

np.save("bpe_input_padded.npy", input_padded)
np.save("bpe_target_padded.npy", target_padded)

print("✅ Encoded and saved input/target sequences for training")

✅ Encoded and saved input/target sequences for training


In [24]:
import sentencepiece as spm
sp = spm.SentencePieceProcessor()
sp.load("bpe_tokenizer.model")
vocab_size = sp.get_piece_size()


In [25]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

embedding_dim = 256
lstm_units = 512

# Encoder
encoder_inputs = Input(shape=(None,), name="encoder_inputs")
encoder_embedding = Embedding(vocab_size, embedding_dim)(encoder_inputs)
_, state_h, state_c = LSTM(lstm_units, return_state=True)(encoder_embedding)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,), name="decoder_inputs")
decoder_embedding = Embedding(vocab_size, embedding_dim)(decoder_inputs)
decoder_lstm = LSTM(lstm_units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
decoder_dense = Dense(vocab_size, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# Final model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 256) │  2,048,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 256) │  2,048,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 512),     │  1,574,912 │ embedding_2[0][0] │
│                     │ (None, 512),      │            │                   │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, None,     │  1,574,912 │ embedding_3[0][0… │
│                     │ 512), (None,      │            │ lstm_2[0][1],     │
│                     │ 512), (None,      │            │ lstm_2[0][2]      │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None,      │  4,104,000 │ lstm_3[0][0]      │
│                     │ 8000)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 11,349,824 (43.30 MB)

 Trainable params: 11,349,824 (43.30 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
import numpy as np

# Load padded BPE-encoded sequences
encoder_input = np.load("bpe_input_padded.npy")
target_sequences = np.load("bpe_target_padded.npy")

# Prepare decoder input/output
decoder_input = target_sequences[:, :-1]
decoder_output = np.expand_dims(target_sequences[:, 1:], -1)



In [27]:
model.fit(
    [encoder_input, decoder_input],
    decoder_output,
    batch_size=64,
    epochs=10,
    validation_split=0.1
)


Epoch 1/10
3258/3258 ━━━━━━━━━━━━━━━━━━━━ 167s 50ms/step - accuracy: 0.7213 - loss: 1.6841 - val_accuracy: 0.7746 - val_loss: 1.1224
Epoch 2/10
3258/3258 ━━━━━━━━━━━━━━━━━━━━ 164s 50ms/step - accuracy: 0.7832 - loss: 1.0581 - val_accuracy: 0.7850 - val_loss: 1.0288
Epoch 3/10
3258/3258 ━━━━━━━━━━━━━━━━━━━━ 164s 50ms/step - accuracy: 0.7915 - loss: 0.9788 - val_accuracy: 0.7900 - val_loss: 0.9828
Epoch 4/10
3258/3258 ━━━━━━━━━━━━━━━━━━━━ 163s 50ms/step - accuracy: 0.7969 - loss: 0.9322 - val_accuracy: 0.7933 - val_loss: 0.9573
Epoch 5/10
3258/3258 ━━━━━━━━━━━━━━━━━━━━ 163s 50ms/step - accuracy: 0.8004 - loss: 0.9037 - val_accuracy: 0.7953 - val_loss: 0.9413
Epoch 6/10
3258/3258 ━━━━━━━━━━━━━━━━━━━━ 163s 50ms/step - accuracy: 0.8028 - loss: 0.8834 - val_accuracy: 0.7966 - val_loss: 0.9300
Epoch 7/10
3258/3258 ━━━━━━━━━━━━━━━━━━━━ 163s 50ms/step - accuracy: 0.8044 - loss: 0.8700 - val_accuracy: 0.7980 - val_loss: 0.9218
Epoch 8/10
3258/3258 ━━━━━━━━━━━━━━━━━━━━ 163s 50ms/step - accuracy: 

In [34]:
model.save("lstm_bpe_recipe_model.keras")


In [37]:
# Already trained earlier
# Just copy the .model file
import shutil
shutil.copy("bpe_tokenizer.model", "lstm_bpe_tokenizer.model")


'lstm_bpe_tokenizer.model'

Testing Phase

In [18]:
import numpy as np
import pandas as pd
import sentencepiece as spm
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


In [41]:
# ✅ Test BPE model after proper train/test split

import numpy as np
import pandas as pd
import sentencepiece as spm
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ----------------------------
# 1. Load Trained Tokenizer and Model
# ----------------------------
sp = spm.SentencePieceProcessor()
sp.load("bpe_tokenizer.model")
model = load_model("lstm_bpe_recipe_model.keras")

# ----------------------------
# 2. Load Test Data (must match your earlier split)
# ----------------------------
from sklearn.model_selection import train_test_split

df = pd.read_csv("preprocessed_recipes.csv")
_, test_df = train_test_split(df, test_size=0.1, random_state=42)
MAX_LEN_INPUT = 132

# ----------------------------
# 3. Rebuild Inference Model
# ----------------------------
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model

vocab_size = sp.get_piece_size()
embedding_dim = 256
lstm_units = 512

# Encoder
encoder_inputs = Input(shape=(None,), name="encoder_inputs")
enc_emb_layer = Embedding(vocab_size, embedding_dim)
encoder_embedding = enc_emb_layer(encoder_inputs)
encoder_lstm_layer = LSTM(lstm_units, return_state=True)  # Remove name='encoder_lstm'
_, state_h_enc, state_c_enc = encoder_lstm_layer(encoder_embedding)
encoder_model = Model(encoder_inputs, [state_h_enc, state_c_enc])

# Decoder
decoder_inputs = Input(shape=(None,), name="decoder_inputs")
decoder_state_input_h = Input(shape=(lstm_units,), name="decoder_h")
decoder_state_input_c = Input(shape=(lstm_units,), name="decoder_c")

decoder_embedding_layer = Embedding(vocab_size, embedding_dim)
decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm_layer = LSTM(lstm_units, return_sequences=True, return_state=True)  # Remove name='decoder_lstm'
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm_layer(
    decoder_embedding, initial_state=[decoder_state_input_h, decoder_state_input_c])

decoder_dense_layer = Dense(vocab_size, activation='softmax')
decoder_outputs = decoder_dense_layer(decoder_outputs)

decoder_model = Model(
    [decoder_inputs, decoder_state_input_h, decoder_state_input_c],
    [decoder_outputs, state_h_dec, state_c_dec]
)

# Assign weights by layer reference using index instead of name
# Assign weights by layer name instead of index
enc_emb_layer.set_weights(model.get_layer('embedding').get_weights())  # Changed to layer name
encoder_lstm_layer.set_weights(model.get_layer('lstm').get_weights())  # Changed to layer name

decoder_embedding_layer.set_weights(model.get_layer('embedding_1').get_weights())  # Changed to layer name
decoder_lstm_layer.set_weights(model.get_layer('lstm_1').get_weights())  # Changed to layer name
decoder_dense_layer.set_weights(model.get_layer('dense').get_weights())  # Changed to layer name





In [42]:
# ----------------------------
# 4. Inference Function
# ----------------------------
def encode_input(text):
    seq = sp.encode(text, out_type=int)
    return pad_sequences([seq], maxlen=MAX_LEN_INPUT, padding='post')

def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = sp.piece_to_id("<start>")

    stop_condition = False
    decoded_tokens = []

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        # Convert sampled_token_index to a regular Python int
        sampled_token_index = int(sampled_token_index)

        sampled_word = sp.id_to_piece(sampled_token_index)

        if sampled_word == "<end>" or len(decoded_tokens) > 300:
            stop_condition = True
        else:
            decoded_tokens.append(sampled_word)

        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]

    return sp.decode(decoded_tokens) # Added this line to actually return the decoded sequence

In [44]:
# 5. Run Predictions and Evaluate BLEU
# ----------------------------
smoothie = SmoothingFunction().method4
bleu_scores = []

for i in range(10):
    input_text = test_df.iloc[i]['input_text']
    true_output = test_df.iloc[i]['steps_text']
    input_seq = encode_input(input_text)
    pred_text = decode_sequence(input_seq)

    reference = [true_output.replace("<start>", "").replace("<end>", "").strip().split()]
    candidate = pred_text.strip().split()

    score = sentence_bleu(reference, candidate, smoothing_function=smoothie)
    bleu_scores.append(score)

    print(f"\n[Sample {i+1}]")
    print("Input:", input_text[:150], "...")
    print("True Steps:", true_output[:200], "...")
    print("Predicted Steps:", pred_text[:200], "...")

avg_bleu = round(sum(bleu_scores) / len(bleu_scores), 4)
print(f"\n✅ Average BLEU score on test set: {avg_bleu}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━